# 04 — Tree Models

**Objective:** compare tree models with the shared training-only evaluation framework.  
**Owner:** Member 02  

> Leakage warning: the reserved test set is never fitted, scored, or inspected here.

In [ ]:
from pathlib import Path
import sys

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'configs/config.yaml').is_file())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from sklearn.tree import DecisionTreeClassifier

from src.data import get_dataset_summary, load_dataset
from src.experiments import run_experiment
from src.modeling import create_random_forest_classifier, describe_estimator
from src.validation import create_train_test_split

X_all, y_all, metadata = load_dataset(optimize_memory=True)
X_train, X_test, y_train, y_test = create_train_test_split(X_all, y_all)
train_summary = get_dataset_summary(X_train, y_train, {**metadata, 'partition': 'train'})
print('Training shape:', X_train.shape)
print('Reserved test shape (not evaluated):', X_test.shape)
print('Training target proportions:', train_summary['target_proportions'])

## Model definitions

The Random Forest uses the shared factory with explicit experimental parameters. `src.modeling` currently has no Decision Tree factory, so that estimator remains explicit here.

In [ ]:
decision_tree = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42)
random_forest = create_random_forest_classifier(
    n_estimators=200, max_depth=8, class_weight='balanced', random_state=42, n_jobs=-1,
)
display(describe_estimator(decision_tree))
display(describe_estimator(random_forest))

## Shared cross-validation experiments

`run_experiment` uses the centrally configured folds and metrics. Results stay in memory and are not registered before review.

In [ ]:
dt_fold_results, dt_summary = run_experiment(
    estimator=decision_tree, X=X_train, y=y_train,
    experiment_id='M02-DT-001', model_name='Decision Tree',
    member='Member 02', branch='feature/eda+tree_models',
)
display(dt_fold_results)
display(pd.DataFrame(dt_summary['metrics']).T)

In [ ]:
rf_fold_results, rf_summary = run_experiment(
    estimator=random_forest, X=X_train, y=y_train,
    experiment_id='M02-RF-001', model_name='Random Forest',
    member='Member 02', branch='feature/eda+tree_models',
)
display(rf_fold_results)
display(pd.DataFrame(rf_summary['metrics']).T)

In [ ]:
comparison = pd.DataFrame({
    'Decision Tree': {name: values['validation_mean'] for name, values in dt_summary['metrics'].items()},
    'Random Forest': {name: values['validation_mean'] for name, values in rf_summary['metrics'].items()},
}).T.sort_values('roc_auc', ascending=False)
display(comparison)

## Tree-model summary

- Both estimators use the same shared training partition, folds, metrics, and ROC-AUC primary criterion.
- Interpret the executed `comparison` table; no scores are hard-coded.
- The reserved final test set remains untouched during model comparison.